# Week 10 — Run a small interaction repeatedly

**Research task:** Compare a no-interaction baseline with a condition in which each simulated actor sees the preceding action.

**Python introduced:** repeated function calls, `range(...)`, accumulated run lists, conditions and small outcome summaries.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session10/session10_collective_intelligence.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.

For local work, download the complete repository rather than this notebook alone, start it with `uv run jupyter lab`, and follow any `NEXT STEP` printed by the setup cell. The full instructions are in `docs/ENVIRONMENT_SETUP.md` and in the course book's computing chapter.


In [ ]:
SESSION = "session10"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Choose a route and define the action schema

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.


In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
action_schema = {
    "type":"object","properties":{"action":{"type":"string"}},
    "required":["action"],"additionalProperties":False,
}


## Define one actor update function

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
def choose_action(actor, observation, route):
    messages = [{"role":"user","content":(
        "Choose JOIN or STAY_OUT for this simulated actor. Actor: " + json.dumps(actor) +
        " Observation: " + observation + " Return JSON."
    )}]
    if route == "openrouter":
        with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
            response = client.chat.send(
                model=HOSTED_MODEL,messages=messages,temperature=0,
                response_format={"type":"json_schema","json_schema":{
                    "name":"join_action","strict":True,"schema":action_schema,
                }},
            )
        raw_output = response.choices[0].message.content
    else:
        response = ollama.chat(think=False, model=LOCAL_MODEL,messages=messages,format=action_schema,options={"temperature":0})
        raw_output = response.message.content
    return json.loads(raw_output)["action"]


## Store three actors and two conditions

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
actors = [
    {"id":"a","initial_support":"low"},
    {"id":"b","initial_support":"medium"},
    {"id":"c","initial_support":"high"},
]
conditions = ["baseline", "interaction"]
all_runs = []

## Repeat each condition three times

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for condition in conditions:
    for run_number in range(3):
        actions = []
        previous_action = "No prior action is visible."
        for actor in actors:
            if condition == "baseline":
                observation = "No other actor's action is visible."
            else:
                observation = "The preceding actor chose: " + previous_action
            action = choose_action(actor, observation, ROUTE)
            actions.append(action)
            previous_action = action
        all_runs.append({"condition":condition,"run":run_number,"actions":actions})
        print(condition, run_number, actions)

## Compare convergence without selecting one favorite run

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for record in all_runs:
    all_same = len(set(record["actions"])) == 1
    print(record["condition"], record["run"], "all same:", all_same)

# ONE CHANGE: reverse the actor order and rerun all conditions.

## Methodological check

Repeated convergence is not automatically collective intelligence. Compare the baseline, preserve order and consider shared model priors, prompt effects and leakage.
## Recording

Explain one actor update, one full group run and how `range(3)` creates repeated evidence. Reverse actor order and discuss what the changed or unchanged outcomes establish.